# Setup From GitHub

Set `REPO_URL` to your GitHub repo. These notebooks avoid reinstalling PyTorch because Kaggle already ships a CUDA-compatible build.

In [ ]:
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd


In [ ]:
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi


# Evaluation / Judging Only

Attach the generation notebook outputs as a Kaggle input dataset, then set `EVAL_DIR`. This notebook does not train or generate model outputs.

In [ ]:
# If pairwise files are in the current repo/session:
EVAL_DIR = "outputs/eval"

# If pairwise files are from an attached Kaggle output dataset, use something like:
# EVAL_DIR = "/kaggle/input/YOUR_GENERATION_OUTPUT_DATASET/outputs/eval"

RUN_LOCAL_JUDGE = False
JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


## Inspect Pairwise Files

In [ ]:
!ls -lah {EVAL_DIR}
!sed -n '1,2p' {EVAL_DIR}/dpo_vs_sft_pairs.jsonl


## Optional Local Judge

Local judging loads another model. If memory is tight, keep this off and download the pairwise JSONL for manual/API judging.

In [ ]:
if RUN_LOCAL_JUDGE:
    !python scripts/judge_pairwise_local.py \
      --pairs_path {EVAL_DIR}/dpo_vs_sft_pairs.jsonl \
      --output_path outputs/eval/dpo_vs_sft_judged.jsonl \
      --judge_model {JUDGE_MODEL}

    !python scripts/judge_pairwise_local.py \
      --pairs_path {EVAL_DIR}/dpo_vs_base_pairs.jsonl \
      --output_path outputs/eval/dpo_vs_base_judged.jsonl \
      --judge_model {JUDGE_MODEL}


## Artifact List

In [ ]:
!find outputs/eval -maxdepth 1 -type f 2>/dev/null | sort || true
